Teste de Fuzzy

In [16]:
caminho_do_arquivo = '/media/juan/SSD_D/ufc/2026.2/instrumentação/Projeto_grupo_4/Projeto/eplusout.csv'

In [17]:
import os

In [18]:
import numpy as np

In [19]:
print(os.getcwd())

/media/juan/SSD_D/ufc/2026.2/instrumentação/Projeto_grupo_4/Projeto


### Definição do LHS Hipercubo Latino

In [20]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [21]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
from scipy.stats import qmc

In [23]:
limites_parametros = {
    'temp_setpoint': [18.0, 26.0],       # Ex: Temperatura (°C)
    'potencia_equip': [1000.0, 5000.0],  # Ex: Carga térmica (W)
    'taxa_infiltracao': [0.1, 1.5],      # Ex: Trocas de ar/h
    'n_ocupantes': [1, 15]               # Ex: Número de pessoas
}

In [24]:
n_amostras = 200
n_var = len(limites_parametros)

In [25]:
# 2. CRIAÇÃO DO HIPERCUBO LATINO (Valores entre 0 e 1)
# d = dimensão (número de parâmetros)
sampler = qmc.LatinHypercube(d=n_var, seed=42)
amostras_normalizadas = sampler.random(n=n_amostras)

In [26]:
# 3. PROJEÇÃO PARA OS LIMITES REAIS
limites_inferiores = [v[0] for v in limites_parametros.values()]
limites_superiores = [v[1] for v in limites_parametros.values()]

# Transforma a amostragem de [0, 1] para os intervalos reais das variáveis
amostras_projetadas = qmc.scale(amostras_normalizadas, limites_inferiores, limites_superiores)

# 4. ESTRUTURAÇÃO DA MATRIZ EM DATAFRAME
df_lhs = pd.DataFrame(amostras_projetadas, columns=limites_parametros.keys())

# Caso alguma variável precise ser inteira (ex: contagem de ocupantes)
df_lhs['n_ocupantes'] = df_lhs['n_ocupantes'].round().astype(int)

# Salvar a matriz de experimentos em um arquivo .csv para alimentar as simulações
df_lhs.to_csv('matriz_lhs_simulacoes.csv', index=False)

print(df_lhs.head())

   temp_setpoint  potencia_equip  taxa_infiltracao  n_ocupantes
0      23.049042     1591.222431          1.192990            5
1      18.196233     2620.487553          1.305672           13
2      18.954875     1850.992281          0.559404            6
3      23.694245     2343.544768          1.125896           10
4      21.937817     3338.723655          0.822207            2


### Definição do Fuzzy Trápezio

In [27]:
temperaturas = np.arange(0, 41, 1)

In [28]:
temp_atual = 30

In [29]:
def fuzzy_trapezoidal(x, a, b, c, d):
    """
    a, d: limites inferiores (pertinência começa/termina em 0)
    b, c: limites do platô (pertinência atinge e se mantém em 1)
    """
    subida = (x - a) / (b - a)
    descida = (d - x) / (d - c)
    # Garante que os limites não ultrapassem o intervalo fechado [0, 1][cite: 3]
    return np.maximum(0, np.minimum(np.minimum(subida, 1), descida))

In [30]:
# 4. EXECUÇÃO DO TESTE FUZZY (FUZZIFICAÇÃO)
# Parametrizando o Trapézio (ex: "Temperatura Agradável", com platô ideal entre 20 e 35 graus)
grau_trapezio = fuzzy_trapezoidal(temp_atual, a=15, b=20, c=35, d=40)

# 5. RESULTADOS
print(f"Entrada exata do sensor (Crisp): {temp_atual}°C")
print(f"Grau de pertinência no modelo Trapezoidal: {grau_trapezio:.2f}")

Entrada exata do sensor (Crisp): 30°C
Grau de pertinência no modelo Trapezoidal: 1.00
